In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path(r"C:\REPLACE_WITH_LOCAL_DATA\Desktop\CCN_Spectrograms")
PAL_DIR = DATA_ROOT / "pal"
PER_DIR = DATA_ROOT / "per"

print("DATA_ROOT exists:", DATA_ROOT.exists(), DATA_ROOT)
print("PAL_DIR exists:", PAL_DIR.exists(), PAL_DIR)
print("PER_DIR exists:", PER_DIR.exists(), PER_DIR)


In [ ]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

def list_subjects(modality_dir: Path):
    return sorted([p for p in modality_dir.iterdir() if p.is_dir()])

def count_images_in_subject(subj_dir: Path):
    return sum(1 for p in subj_dir.rglob("*") if p.suffix.lower() in IMG_EXTS)

pal_subjects = list_subjects(PAL_DIR)
per_subjects = list_subjects(PER_DIR)

print("PAL subjects:", len(pal_subjects))
print("PER subjects:", len(per_subjects))

# show first 5 subject folder names
print("\nFirst 5 PAL subjects:", [p.name for p in pal_subjects[:5]])
print("First 5 PER subjects:", [p.name for p in per_subjects[:5]])

# image counts summary
pal_counts = [count_images_in_subject(s) for s in pal_subjects]
per_counts = [count_images_in_subject(s) for s in per_subjects]

print("\nPAL images per subject: min/mean/max =", min(pal_counts), sum(pal_counts)/len(pal_counts), max(pal_counts))
print("PER images per subject: min/mean/max =", min(per_counts), sum(per_counts)/len(per_counts), max(per_counts))

print("\nTotal PAL images:", sum(pal_counts))
print("Total PER images:", sum(per_counts))


In [ ]:
import random

def sample_filenames(modality_dir: Path, n_subjects=3, n_files=10):
    subjects = list_subjects(modality_dir)
    pick = random.sample(subjects, min(n_subjects, len(subjects)))
    out = {}
    for s in pick:
        files = [p for p in s.rglob("*") if p.suffix.lower() in IMG_EXTS]
        files = sorted(files)
        if len(files) == 0:
            continue
        out[s.name] = [f.name for f in random.sample(files, min(n_files, len(files)))]
    return out

print("Sample PAL filenames:")
pal_samples = sample_filenames(PAL_DIR, n_subjects=3, n_files=12)
for subj, names in pal_samples.items():
    print("\n", subj)
    for nm in names:
        print("  ", nm)

print("\n\nSample PER filenames:")
per_samples = sample_filenames(PER_DIR, n_subjects=3, n_files=12)
for subj, names in per_samples.items():
    print("\n", subj)
    for nm in names:
        print("  ", nm)


In [ ]:
import csv

OUT_CSV = DATA_ROOT / "labels_to_fill.csv"

rows = []
for modality, mdir in [("pal", PAL_DIR), ("per", PER_DIR)]:
    for subj_dir in list_subjects(mdir):
        subject_id = subj_dir.name
        for img_path in subj_dir.rglob("*"):
            if img_path.suffix.lower() in IMG_EXTS:
                rows.append([subject_id, modality, str(img_path), ""])  # region blank

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["subject_id", "modality", "image_path", "region_label"])
    w.writerows(rows)

print("✅ CSV created:", OUT_CSV)
print("Rows:", len(rows))
print("Min: open it in Excel, fill region_label with 8 region names, save.")


In [ ]:
import pandas as pd
import re
from pathlib import PureWindowsPath

csv_path = (r"C:\REPLACE_WITH_LOCAL_DATA\Desktop\CCN_Spectrograms/labels_to_fill.csv")  # uploaded file in this chat
df = pd.read_csv(csv_path)

def parse_pair(path):
    name = PureWindowsPath(path).name
    m = re.search(r"_spectrogram_(\d+)_(\d+)\.png$", name)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))

df["a"], df["b"] = zip(*df["image_path"].map(parse_pair))
print("Unique a values:", sorted(df["a"].dropna().unique()))
print("Unique b range:", int(df["b"].min()), "to", int(df["b"].max()))
print("\nCounts per a:")
print(df["a"].value_counts().sort_index())


In [ ]:
!pip install -q torch torchvision timm scikit-learn pillow


In [ ]:
import os, random, shutil
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torchvision import transforms
import timm
from sklearn.cluster import KMeans

DATA_ROOT = Path(r"C:\REPLACE_WITH_LOCAL_DATA\Desktop\CCN_Spectrograms")
OUT_ROOT = DATA_ROOT / "cluster_preview"
OUT_ROOT.mkdir(exist_ok=True)

IMG_EXTS = {".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}

# Collect a subset first (to keep it fast). We'll start with 2000 images.
all_imgs = []
for modality in ["pal","per"]:
    for subj in (DATA_ROOT/modality).iterdir():
        if subj.is_dir():
            for p in subj.rglob("*"):
                if p.suffix.lower() in IMG_EXTS:
                    all_imgs.append((modality, subj.name, p))

print("Total images found:", len(all_imgs))

random.seed(42)
sample_n = 2000  # start small; later we can scale up
sample = random.sample(all_imgs, sample_n)
print("Using sample:", len(sample))

# Pretrained feature extractor (fast + strong)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = timm.create_model("resnet50", pretrained=True, num_classes=0, global_pool="avg").to(device)
model.eval()

tfm = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

@torch.no_grad()
def embed_image(path):
    img = Image.open(path).convert("RGB")
    x = tfm(img).unsqueeze(0).to(device)
    feat = model(x).squeeze(0).cpu().numpy()
    return feat

# Extract embeddings
X = []
meta = []
for modality, subj, path in sample:
    X.append(embed_image(path))
    meta.append((modality, subj, str(path)))

X = np.vstack(X)
print("Embeddings shape:", X.shape)

# KMeans clustering into 8 groups
k = 8
km = KMeans(n_clusters=k, random_state=42, n_init="auto")
clusters = km.fit_predict(X)
print("Cluster counts:", np.bincount(clusters))

# Save preview images per cluster
for c in range(k):
    (OUT_ROOT / f"cluster_{c}").mkdir(parents=True, exist_ok=True)

# Copy 25 examples per cluster for you to inspect
per_cluster = 25
picked = {c:0 for c in range(k)}
for (modality, subj, p), c in zip(sample, clusters):
    if picked[c] < per_cluster:
        dst = OUT_ROOT / f"cluster_{c}" / f"{modality}_{subj}_{Path(p).name}"
        shutil.copy(p, dst)
        picked[c] += 1

print("✅ Saved preview folders here:", OUT_ROOT)
print("Open them and tell me what anatomical region each cluster corresponds to.")


In [ ]:
# If you already have these, you can skip this cell.
!pip -q install tensorflow scikit-learn pandas matplotlib pillow


In [ ]:
# # Cell 1 — Paths + basic checks

In [ ]:
import os, re, random
import numpy as np
import pandas as pd

DATA_ROOT = r"C:\REPLACE_WITH_LOCAL_DATA\Desktop\CCN_Spectrograms"
PAL_DIR = os.path.join(DATA_ROOT, "pal")
PER_DIR = os.path.join(DATA_ROOT, "per")

print("DATA_ROOT exists:", os.path.exists(DATA_ROOT), DATA_ROOT)
print("PAL_DIR exists:", os.path.exists(PAL_DIR), PAL_DIR)
print("PER_DIR exists:", os.path.exists(PER_DIR), PER_DIR)

assert os.path.isdir(PAL_DIR), "pal folder not found"
assert os.path.isdir(PER_DIR), "per folder not found"


In [ ]:
# Cell 2 — Build an index (image path, subject, modality label)

In [ ]:
from pathlib import Path

def list_subjects(folder):
    return sorted([d.name for d in Path(folder).iterdir() if d.is_dir()])

pal_subjects = list_subjects(PAL_DIR)
per_subjects = list_subjects(PER_DIR)

print("PAL subjects:", len(pal_subjects))
print("PER subjects:", len(per_subjects))
print("First 5 PAL subjects:", pal_subjects[:5])
print("First 5 PER subjects:", per_subjects[:5])

# Intersection check (should match)
common = sorted(set(pal_subjects).intersection(set(per_subjects)))
print("Common subjects:", len(common))
assert len(common) > 0, "No common subjects between pal and per"

def collect_rows(root_dir, modality_name, label_value):
    rows = []
    root = Path(root_dir)
    for subj_dir in root.iterdir():
        if not subj_dir.is_dir():
            continue
        subject = subj_dir.name
        for img_path in subj_dir.glob("*.png"):
            rows.append({
                "path": str(img_path),
                "subject": subject,
                "modality": modality_name,
                "label": int(label_value)
            })
    return rows

rows = []
rows += collect_rows(PAL_DIR, "pal", 0)
rows += collect_rows(PER_DIR, "per", 1)

df = pd.DataFrame(rows)
print("Total images:", len(df))
print(df["modality"].value_counts())
print("Unique subjects:", df["subject"].nunique())

# quick sanity check
print("Example rows:")
display(df.sample(5, random_state=42))


In [ ]:
# Cell 3 — Subject-wise split (train/val/test)

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

subjects = sorted(df["subject"].unique())

# 70% train, 15% val, 15% test by subject
train_subj, temp_subj = train_test_split(subjects, test_size=0.30, random_state=SEED)
val_subj, test_subj   = train_test_split(temp_subj, test_size=0.50, random_state=SEED)

train_df = df[df["subject"].isin(train_subj)].reset_index(drop=True)
val_df   = df[df["subject"].isin(val_subj)].reset_index(drop=True)
test_df  = df[df["subject"].isin(test_subj)].reset_index(drop=True)

print("Subjects -> train/val/test:", len(train_subj), len(val_subj), len(test_subj))
print("Images   -> train/val/test:", len(train_df), len(val_df), len(test_df))
print("Train modality counts:\n", train_df["modality"].value_counts())
print("Val modality counts:\n", val_df["modality"].value_counts())
print("Test modality counts:\n", test_df["modality"].value_counts())

# Safety: no subject overlap
assert set(train_subj).isdisjoint(val_subj)
assert set(train_subj).isdisjoint(test_subj)
assert set(val_subj).isdisjoint(test_subj)


In [ ]:
# Cell 4 — TensorFlow dataset loader (fast, no RAM blow-up)

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)   # strong default; change to (128,128) if you want faster
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def decode_img(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)              # grayscale
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img

def augment(img):
    # light augmentations for robustness
    img = tf.image.random_brightness(img, max_delta=0.08)
    img = tf.image.random_contrast(img, lower=0.90, upper=1.10)
    return tf.clip_by_value(img, 0.0, 1.0)

def make_ds(frame, training=False):
    paths = frame["path"].values
    labels = frame["label"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(2000, seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(lambda p, y: (decode_img(p), y), num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.map(lambda x, y: (augment(x), y), num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df, training=False)
test_ds  = make_ds(test_df, training=False)

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Val batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:
# Cell 5 — Build a strong CNN baseline (MobileNetV2)

In [ ]:
from tensorflow.keras import layers, models

# Convert 1-channel -> 3-channel to use ImageNet backbone
def to_rgb(x):
    return tf.image.grayscale_to_rgb(x)

base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet"
)
base.trainable = False  # start frozen for stable training

inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 1))
x = layers.Lambda(to_rgb)(inputs)

# MobileNetV2 preprocessing
x = tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)

x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="acc"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()


In [ ]:
# Cell 6 — Train (frozen backbone)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=2, min_lr=1e-6),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# Cell 7 — Optional fine-tuning (usually improves)

###############################
# is ko run nahi krna

In [ ]:
# Unfreeze last ~30 layers for fine-tuning
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="acc"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# Cell 8 — Evaluate on test + confusion matrix + ROC-AUC

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# predict probabilities
y_true = test_df["label"].values
y_prob = model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Test ROC-AUC:", roc_auc_score(y_true, y_prob))
print("\nClassification report:\n", classification_report(y_true, y_pred, target_names=["pal","per"]))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:\n", cm)

# Plot confusion matrix (simple)
plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix (pal vs per)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks([0,1], ["pal","per"])
plt.yticks([0,1], ["pal","per"])
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.show()

# ROC curve
fpr, tpr, thr = roc_curve(y_true, y_prob)
plt.figure()
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1], linestyle="--")
plt.title("ROC Curve (pal vs per)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()


In [ ]:
# Cell 9 — Save model + save a results CSV (for your paper later)

In [ ]:
OUT_DIR = os.path.join(DATA_ROOT, "optionB_results")
os.makedirs(OUT_DIR, exist_ok=True)

model_path = os.path.join(OUT_DIR, "pal_vs_per_mobilenetv2.h5")
model.save(model_path)
print("✅ Saved model:", model_path)

pred_path = os.path.join(OUT_DIR, "test_predictions.csv")
pd.DataFrame({
    "path": test_df["path"].values,
    "subject": test_df["subject"].values,
    "modality": test_df["modality"].values,
    "y_true": y_true,
    "y_prob": y_prob,
    "y_pred": y_pred
}).to_csv(pred_path, index=False)
print("✅ Saved predictions:", pred_path)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

# =========================================================
# 1) Output folder (Baseline results)
# =========================================================
OUT_DIR = r"C:\REPLACE_WITH_LOCAL_DATA\Desktop\CCN_Spectrograms\optionB_results\MobileNetV2"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# 2) Predict on test set (subject-wise test split)
# =========================================================
y_true = test_df["label"].values  # from your earlier cells
y_prob = model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# =========================================================
# 3) Metrics
# =========================================================
auc = roc_auc_score(y_true, y_prob)
cm = confusion_matrix(y_true, y_pred)
report = classification_report(
    y_true, y_pred,
    target_names=["pal", "per"],
    digits=4
)

print("Test ROC-AUC:", auc)
print("\nClassification report:\n", report)
print("Confusion matrix:\n", cm)

# =========================================================
# 4) Save predictions CSV
# =========================================================
pred_df = pd.DataFrame({
    "y_true": y_true,
    "y_prob": y_prob,
    "y_pred": y_pred
})
csv_path = os.path.join(OUT_DIR, "baseline_test_predictions.csv")
pred_df.to_csv(csv_path, index=False)

# =========================================================
# 5) Save metrics TXT
# =========================================================
txt_path = os.path.join(OUT_DIR, "baseline_metrics.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"ROC-AUC: {auc:.6f}\n\n")
    f.write("Classification Report:\n")
    f.write(report + "\n\n")
    f.write("Confusion Matrix:\n")
    f.write(np.array2string(cm))

# =========================================================
# 6) Save Confusion Matrix figure (PNG, 300 DPI)
# =========================================================
plt.figure(figsize=(5, 5))
plt.imshow(cm, cmap="viridis")
plt.title("Confusion Matrix (Baseline CNN: pal vs per)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks([0, 1], ["pal", "per"])
plt.yticks([0, 1], ["pal", "per"])

for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center", color="white", fontsize=12)

plt.tight_layout()
cm_fig_path = os.path.join(OUT_DIR, "baseline_confusion_matrix.png")
plt.savefig(cm_fig_path, dpi=300)
plt.close()

# =========================================================
# 7) Save ROC Curve figure (PNG, 300 DPI)
# =========================================================
fpr, tpr, _ = roc_curve(y_true, y_prob)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve (Baseline CNN: pal vs per)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()

roc_fig_path = os.path.join(OUT_DIR, "baseline_roc_curve.png")
plt.savefig(roc_fig_path, dpi=300)
plt.close()

# =========================================================
# 8) Save model (recommended .keras + optional .h5)
# =========================================================
keras_path = os.path.join(OUT_DIR, "baseline_mobilenetv2.keras")
model.save(keras_path)

# Optional legacy format (keep if you want)
h5_path = os.path.join(OUT_DIR, confirm_name if False else "baseline_mobilenetv2.h5")
# If you want H5 too, change False -> True manually below:
# model.save(os.path.join(OUT_DIR, "baseline_mobilenetv2.h5"))

print("\n✅ Saved baseline artifacts to:")
print(" -", OUT_DIR)
print("✅ Files:")
print(" -", os.path.basename(csv_path))
print(" -", os.path.basename(txt_path))
print(" -", os.path.basename(cm_fig_path))
print(" -", os.path.basename(roc_fig_path))
print(" -", os.path.basename(keras_path))
